In [1]:
# ============================================================
# Cell 0: Install all dependencies (run once per session)
# ============================================================
!pip install -q torch torchvision transformers facenet-pytorch \
               opencv-python pillow numpy mediapipe huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 143.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.5 MB/s eta 0:00:00


In [2]:
# ============================================================
# Cell 1: Download MediaPipe face landmarker model
# ============================================================
import os
if not os.path.exists("face_landmarker.task"):
    !wget -q https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
    print("✅ face_landmarker.task downloaded")
else:
    print("✅ face_landmarker.task already present")


✅ face_landmarker.task downloaded


In [3]:
# ============================================================
# Cell 2: Imports & Model Setup
# ============================================================
import torch
import cv2
import numpy as np
from PIL import Image, ImageFilter
from facenet_pytorch import MTCNN
from transformers import pipeline
from collections import deque
import base64

from IPython.display import display
from google.colab.output import eval_js
import ipywidgets as widgets

# ── Device ────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Face detector ─────────────────────────────────────────────
mtcnn = MTCNN(image_size=224, margin=20, device=device)

# ── Emotion model ─────────────────────────────────────────────
emotion_model = pipeline(
    "image-classification",
    model="abhilash88/face-emotion-detection",
    device=0 if device == 'cuda' else -1
)

# ── Race / ethnicity model ────────────────────────────────────
race_model = pipeline(
    "image-classification",
    model="cledoux42/Ethnicity_Test_v003",
    device=0 if device == 'cuda' else -1
)

# ── Label / colour maps ───────────────────────────────────────
EMOTION_LABELS = {
    "LABEL_0": "Angry",   "LABEL_1": "Disgust", "LABEL_2": "Fear",
    "LABEL_3": "Happy",   "LABEL_4": "Sad",     "LABEL_5": "Surprise",
    "LABEL_6": "Neutral",
}
EMOTION_COLORS = {
    "Angry":   (0, 0, 255),   "Disgust":  (0, 140, 0),
    "Fear":    (128, 0, 128), "Happy":    (0, 215, 255),
    "Sad":     (255, 100, 0), "Surprise": (0, 165, 255),
    "Neutral": (180, 180, 180),
}
# Keys must exactly match lowercased labels from cledoux42/Ethnicity_Test_v003
RACE_COLORS = {
    "african":    (180, 100, 255),   # model label: "African"
    "asian":      (255, 200,   0),   # model label: "Asian"
    "caucasian":  (100, 200, 255),   # model label: "Caucasian"
    "hispanic":   (  0, 220, 120),   # model label: "Hispanic"
    "indian":     (  0, 180, 255),   # model label: "Indian"
    # fallback for any unexpected label:
    "black":           (180, 100, 255),
    "latino hispanic": (  0, 220, 120),
    "middle eastern":  (255, 140,   0),
    "white":           (100, 200, 255),
}

# ── Temporal smoothing (PER FACE ID) ──────────────────────────
# Each face gets its own history buffer so that predictions
# for different people never mix together.
from collections import defaultdict

emotion_history_per_face = defaultdict(lambda: deque(maxlen=10))
race_history_per_face    = defaultdict(lambda: deque(maxlen=8))

def smooth_emotion(prediction, face_id=0):
    emotion_history_per_face[face_id].append(prediction)
    counts = {}
    for e in emotion_history_per_face[face_id]:
        counts[e] = counts.get(e, 0) + 1
    return max(counts, key=counts.get)

def smooth_race(label, face_id=0):
    race_history_per_face[face_id].append(label)
    counts = {}
    for r in race_history_per_face[face_id]:
        counts[r] = counts.get(r, 0) + 1
    return max(counts, key=counts.get)

# ── Thresholds ────────────────────────────────────────────────
EMOTION_CONF_THRESHOLD = 0.40
RACE_CONF_THRESHOLD    = 0.35
QUALITY_THRESHOLD      = 0.60

print(f"✅ All models loaded on {device}")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/918 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/344M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

✅ All models loaded on cuda


In [4]:
# ============================================================
# Cell 3: Webcam Helper
# Defines camera functions. Camera is initialised in Cell 7.
# ============================================================
from google.colab.output import eval_js
import base64, cv2, numpy as np

def _init_camera():
    """
    Initialises the webcam stream in the browser.
    Always call this at the start of Cell 7 — Colab resets
    the JS context (window.*) between cell executions so
    window._emotionVideo cannot be relied on to persist.
    """
    result = eval_js("""
    new Promise(async (resolve) => {
        try {
            // Release any existing stream first
            if (window._emotionVideo) {
                const tracks = window._emotionVideo.srcObject
                               ?.getTracks() || [];
                tracks.forEach(t => t.stop());
                window._emotionVideo = null;
            }
            const video = document.createElement('video');
            video.setAttribute('playsinline', '');
            const stream = await navigator.mediaDevices.getUserMedia({
                video: { width: 640, height: 480 }
            });
            video.srcObject = stream;
            await video.play();
            await new Promise(r => setTimeout(r, 1500));
            window._emotionVideo = video;
            window._stopLoop     = false;
            resolve('OK');
        } catch (err) {
            resolve('ERROR: ' + err.message);
        }
    })
    """)
    return result


def capture_frame_from_browser():
    """
    Captures one JPEG frame from the running webcam stream.
    Returns None when window._stopLoop is true (Stop clicked).
    """
    data_url = eval_js("""
    new Promise(async (resolve) => {
        if (!window._emotionVideo || window._stopLoop) {
            resolve('STOP');
            return;
        }
        // Wait for valid dimensions if stream just started
        let w = 0;
        while ((!window._emotionVideo.videoWidth) && w < 2000) {
            await new Promise(r => setTimeout(r, 100));
            w += 100;
        }
        const v = window._emotionVideo;
        const c = document.createElement('canvas');
        c.width  = v.videoWidth  || 640;
        c.height = v.videoHeight || 480;
        c.getContext('2d').drawImage(v, 0, 0);
        resolve(c.toDataURL('image/jpeg', 0.85));
    })
    """)
    if not data_url or data_url == 'STOP':
        return None
    _, encoded = data_url.split(',', 1)
    img_bytes  = base64.b64decode(encoded)
    img_array  = np.frombuffer(img_bytes, dtype=np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)


def inject_stop_button():
    from IPython.display import display, HTML
    display(HTML("""
    <button
      onclick="
        window._stopLoop = true;
        this.textContent = '⏹ Stopping…';
        this.style.background = '#888';
        this.disabled = true;"
      style="padding:10px 28px;font-size:16px;font-weight:bold;
             background:#e53935;color:white;border:none;
             border-radius:6px;cursor:pointer;margin:8px 0;">
      ⏹ Stop
    </button>
    """))


print("✅ Webcam helper functions defined.")
print("   Camera will be initialised when you run Cell 7.")


✅ Webcam helper functions defined.
   Camera will be initialised when you run Cell 7.


In [5]:
# ============================================================
# Cell 4: Advanced CV Preprocessing & Feature Extraction Pipeline
#
# ┌─────────────────────────────────────────────────────────┐
# │  COMPUTER VISION TECHNIQUES USED IN THIS CELL          │
# │                                                         │
# │  IMAGING:                                               │
# │   1. Image Formation — camera model, focal length est.  │
# │   2. Image Sensing — noise estimation via Laplacian     │
# │   3. Binary Images — Otsu thresholding for face masks   │
# │   4. Image Processing I — CLAHE, gamma, white balance   │
# │   5. Image Processing II — bilateral filter, unsharp    │
# │                                                         │
# │  FEATURES:                                              │
# │   6. Edge Detection — Sobel gradients + Canny edges     │
# │   7. Boundary Detection — contour extraction & analysis │
# │   8. SIFT Detector — keypoint density on face region    │
# │   9. Face Detection — MTCNN (Cell 2) + landmarks here   │
# │                                                         │
# │  RECONSTRUCTION I:                                      │
# │  10. Radiometry & Reflectance — mean reflectance est.   │
# │  11. Shape from Shading — surface normals from grads    │
# │  12. Depth from Defocus — Laplacian variance as proxy   │
# │                                                         │
# │  RECONSTRUCTION II:                                     │
# │  13. Optical Flow — Lucas-Kanade sparse flow on lmks    │
# │  14. Camera Calibration — focal length from face width  │
# │                                                         │
# │  PERCEPTION:                                            │
# │  15. Image Segmentation — skin color seg in YCrCb       │
# │  16. Appearance Matching — histogram-based face ReID    │
# │  17. Object Tracking — centroid tracker for face IDs    │
# └─────────────────────────────────────────────────────────┘
# ============================================================
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

base_options = mp_python.BaseOptions(model_asset_path='face_landmarker.task')
options = mp_vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    num_faces=1
)
face_mesh = mp_vision.FaceLandmarker.create_from_options(options)

# ── Landmark indices ──────────────────────────────────────────
LEFT_EYE_IDX   = [362, 385, 387, 263, 373, 380]
RIGHT_EYE_IDX  = [33,  160, 158, 133, 153, 144]
LEFT_BROW_IDX  = [336, 296, 334, 293, 300]
RIGHT_BROW_IDX = [70,   63, 105,  66, 107]
MOUTH_IDX      = [61,  291,  13,  14,  17,   0,  37, 267]
NOSE_TIP_IDX   = 1
FACE_OVAL_IDX  = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323,
                  361, 288, 397, 365, 379, 378, 400, 377, 152, 148,
                  176, 149, 150, 136, 172, 58,  132, 93,  234, 127,
                  162, 21,  54,  103, 67,  109]

# ── State for temporal CV techniques ─────────────────────────
_prev_gray = None          # for Optical Flow (CV Technique 13)
_prev_landmarks = None     # for landmark-based flow
_face_histograms = {}      # for Appearance Matching (CV Technique 16)
_face_centroids = {}       # for Object Tracking (CV Technique 17)
_next_face_id = 0

# ── Shared safe-copy helper ───────────────────────────────────
def _safe(img):
    if img is None:
        raise ValueError("_safe() received None")
    arr = np.ascontiguousarray(img, dtype=np.uint8)
    if arr.ndim != 3 or arr.shape[2] != 3:
        raise ValueError(f"Expected HxWx3, got {arr.shape}")
    return arr

# =====================================================================
# CV TECHNIQUE 4 & 5: Image Processing I & II
# =====================================================================

def apply_clahe(face_rgb, clip_limit=2.5, tile_grid=(4, 4)):
    """[CV: Image Processing I — CLAHE]
    Adaptive histogram equalisation on L channel (LAB space).
    Normalises local contrast without blowing out bright regions."""
    img = _safe(face_rgb)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

def apply_gamma(face_rgb, gamma=1.2):
    """[CV: Image Processing I — Gamma / Radiometric Correction]
    Power-law transform to lift shadows or recover highlights."""
    img   = _safe(face_rgb)
    inv   = 1.0 / gamma
    table = np.array([(i / 255.0) ** inv * 255
                      for i in range(256)], dtype=np.uint8)
    return cv2.LUT(img, table)

def apply_sharpen(face_rgb):
    """[CV: Image Processing II — Unsharp Mask / High-Pass Enhancement]
    Adds high-frequency residual back to sharpen emotion-relevant edges."""
    img       = _safe(face_rgb)
    blur      = cv2.GaussianBlur(img, (5, 5), sigmaX=1.0)
    sharpened = cv2.addWeighted(img, 1.5, blur, -0.5, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)

def apply_bilateral(face_rgb, d=7, sigma_color=50, sigma_space=50):
    """[CV: Image Processing II — Bilateral Filter]
    Edge-preserving noise reduction. Smooths skin without blurring
    eye/lip boundaries."""
    img = _safe(face_rgb)
    h_b, w_b = img.shape[:2]
    if h_b < d + 1 or w_b < d + 1:
        return img
    result = cv2.bilateralFilter(img, d, sigma_color, sigma_space)
    return result if result is not None else img

def apply_white_balance(face_rgb):
    """[CV: Image Processing I — Grey-World White Balance]
    Scales RGB channels so their means are equal. Removes colour casts
    from warm/cool artificial lighting."""
    img    = _safe(face_rgb)
    result = img.astype(np.float32)
    means  = [np.mean(result[:, :, c]) for c in range(3)]
    avg    = np.mean(means)
    for c in range(3):
        if means[c] > 1e-3:
            result[:, :, c] = np.clip(
                result[:, :, c] * (avg / means[c]), 0, 255
            )
    return result.astype(np.uint8)

# =====================================================================
# CV TECHNIQUE 3: Binary Images — Otsu Thresholding
# =====================================================================

def compute_binary_face_mask(face_rgb):
    """[CV: Binary Images — Otsu Thresholding]
    Converts face crop to binary mask using Otsu's method on the
    grayscale image. Used to estimate face area ratio and separate
    foreground from background in the crop."""
    img = _safe(face_rgb)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    face_ratio = np.count_nonzero(binary) / max(1, binary.size)
    return binary, face_ratio

# =====================================================================
# CV TECHNIQUE 6: Edge Detection — Sobel & Canny
# =====================================================================

def compute_edge_features(face_rgb):
    """[CV: Edge Detection — Sobel Gradient Magnitude + Canny]
    Computes:
      - Sobel gradient magnitude (avg & max) — measures texture complexity
      - Canny edge density — ratio of edge pixels to total pixels
    These metrics drive art complexity: more facial texture → denser art."""
    img  = _safe(face_rgb)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # Sobel gradients (Image Processing / Edge Detection)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    # Gradient direction for Shape from Shading (Technique 11)
    direction = np.arctan2(sobel_y, sobel_x)
    # Canny edge detection
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.count_nonzero(edges) / max(1, edges.size)
    return {
        "sobel_mag_mean": float(np.mean(magnitude)),
        "sobel_mag_max":  float(np.max(magnitude)),
        "edge_density":   float(edge_density),
        "gradient_dir":   direction,      # used by Shape from Shading
        "gradient_mag":   magnitude,      # used by Shape from Shading
        "canny_edges":    edges,          # used by Boundary Detection
    }

# =====================================================================
# CV TECHNIQUE 7: Boundary Detection — Contour Extraction
# =====================================================================

def compute_boundary_features(canny_edges):
    """[CV: Boundary Detection — Contour Analysis]
    Extracts contours from Canny edge map. Computes:
      - Number of contours (facial feature complexity)
      - Average contour area (scale of facial features)
      - Contour hierarchy depth (nested feature structures)
    These feed into art pattern density."""
    contours, hierarchy = cv2.findContours(
        canny_edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
    )
    n_contours = len(contours)
    if n_contours == 0:
        return {"n_contours": 0, "avg_contour_area": 0.0, "max_depth": 0}
    areas = [cv2.contourArea(c) for c in contours]
    avg_area = float(np.mean(areas)) if areas else 0.0
    # Hierarchy depth: how many levels of nested contours
    max_depth = 0
    if hierarchy is not None:
        for i in range(len(hierarchy[0])):
            depth = 0
            parent = hierarchy[0][i][3]
            while parent != -1:
                depth += 1
                parent = hierarchy[0][parent][3]
            max_depth = max(max_depth, depth)
    return {
        "n_contours":       n_contours,
        "avg_contour_area": avg_area,
        "max_depth":        max_depth,
    }

# =====================================================================
# CV TECHNIQUE 8: SIFT Detector — Keypoint Density
# =====================================================================

def compute_sift_features(face_rgb):
    """[CV: SIFT Detector — Scale-Invariant Feature Transform]
    Detects SIFT keypoints on the face crop. Metrics:
      - Keypoint count — texture richness
      - Average keypoint size — dominant scale of features
      - Spatial distribution std — whether features are clustered or spread
    Feeds into art detail level: more keypoints → finer art detail."""
    img  = _safe(face_rgb)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    sift = cv2.SIFT_create(nfeatures=200)
    kps, descriptors = sift.detectAndCompute(gray, None)
    n_kps = len(kps)
    if n_kps == 0:
        return {"sift_count": 0, "sift_avg_size": 0.0, "sift_spread": 0.0}
    sizes = [kp.size for kp in kps]
    xs = [kp.pt[0] for kp in kps]
    ys = [kp.pt[1] for kp in kps]
    spread = float(np.std(xs) + np.std(ys))
    return {
        "sift_count":    n_kps,
        "sift_avg_size": float(np.mean(sizes)),
        "sift_spread":   spread,
    }

# =====================================================================
# CV TECHNIQUE 10: Radiometry & Reflectance
# =====================================================================

def compute_reflectance_features(face_rgb):
    """[CV: Radiometry & Reflectance — Mean Reflectance Estimation]
    Estimates apparent reflectance by computing per-channel mean
    intensity (assuming uniform illumination after white balance).
    Also computes specular highlight ratio using thresholding.
    Feeds into art brightness/glow intensity."""
    img = _safe(face_rgb)
    means = [float(np.mean(img[:,:,c])) for c in range(3)]
    luminance = 0.2126 * means[0] + 0.7152 * means[1] + 0.0722 * means[2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # Specular highlights: pixels > 230
    specular_ratio = float(np.count_nonzero(gray > 230)) / max(1, gray.size)
    return {
        "mean_luminance":  luminance / 255.0,
        "specular_ratio":  specular_ratio,
        "channel_means":   means,
    }

# =====================================================================
# CV TECHNIQUE 11: Shape from Shading — Surface Normal Estimation
# =====================================================================

def compute_shape_from_shading(gradient_mag, gradient_dir):
    """[CV: Shape from Shading — Surface Normal Estimation]
    Estimates a simplified surface normal field from image gradients
    (Sobel). Computes:
      - Mean surface curvature proxy (Laplacian of gradient magnitude)
      - Dominant shading direction (average gradient angle)
    These modulate art lighting direction and curvature of shapes."""
    if gradient_mag is None:
        return {"curvature": 0.0, "dominant_angle": 0.0}
    # Laplacian of gradient magnitude as curvature proxy
    mag_uint8 = np.clip(gradient_mag, 0, 255).astype(np.uint8)
    laplacian = cv2.Laplacian(mag_uint8, cv2.CV_64F)
    curvature = float(np.mean(np.abs(laplacian)))
    # Dominant gradient direction (weighted by magnitude)
    weights = gradient_mag / (np.sum(gradient_mag) + 1e-9)
    sin_avg = float(np.sum(np.sin(gradient_dir) * weights))
    cos_avg = float(np.sum(np.cos(gradient_dir) * weights))
    dominant_angle = float(np.arctan2(sin_avg, cos_avg))
    return {
        "curvature":      curvature,
        "dominant_angle":  dominant_angle,
    }

# =====================================================================
# CV TECHNIQUE 12: Depth from Defocus — Laplacian Variance
# =====================================================================

def compute_focus_measure(face_rgb):
    """[CV: Depth from Defocus — Laplacian Variance]
    Laplacian variance measures image sharpness. In-focus regions
    have high variance; blurred/defocused regions have low.
    Acts as a depth proxy: sharper face → closer/more in-focus.
    Used to gate processing quality and modulate art clarity."""
    img  = _safe(face_rgb)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lap  = cv2.Laplacian(gray, cv2.CV_64F)
    focus_measure = float(np.var(lap))
    return focus_measure

# =====================================================================
# CV TECHNIQUE 13: Optical Flow — Lucas-Kanade Sparse Flow
# =====================================================================

def compute_optical_flow(gray_frame, landmarks_px):
    """[CV: Optical Flow — Lucas-Kanade Sparse Flow on Landmarks]
    Tracks facial landmark positions between consecutive frames.
    Computes mean motion magnitude → drives art animation speed.
    High motion = fast art; low motion = calm/breathing art."""
    global _prev_gray, _prev_landmarks
    if landmarks_px is None or len(landmarks_px) < 10:
        _prev_gray = gray_frame.copy() if gray_frame is not None else None
        _prev_landmarks = landmarks_px
        return {"flow_magnitude": 0.0, "flow_direction": 0.0}

    if _prev_gray is None or _prev_landmarks is None:
        _prev_gray = gray_frame.copy()
        _prev_landmarks = landmarks_px
        return {"flow_magnitude": 0.0, "flow_direction": 0.0}

    # Use a subset of landmarks as tracking points
    key_idx = LEFT_EYE_IDX + RIGHT_EYE_IDX + MOUTH_IDX + [NOSE_TIP_IDX]
    key_idx = [i for i in key_idx if i < len(_prev_landmarks) and i < len(landmarks_px)]
    if len(key_idx) < 4:
        _prev_gray = gray_frame.copy()
        _prev_landmarks = landmarks_px
        return {"flow_magnitude": 0.0, "flow_direction": 0.0}

    prev_pts = np.array([_prev_landmarks[i] for i in key_idx], dtype=np.float32).reshape(-1, 1, 2)
    curr_pts = np.array([landmarks_px[i] for i in key_idx], dtype=np.float32).reshape(-1, 1, 2)

    # Lucas-Kanade optical flow
    try:
        next_pts, status, err = cv2.calcOpticalFlowPyrLK(
            _prev_gray, gray_frame, prev_pts, None,
            winSize=(15, 15), maxLevel=2,
            criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
        )
        if next_pts is not None and status is not None:
            good_mask = status.flatten() == 1
            if np.any(good_mask):
                dx = (next_pts[good_mask] - prev_pts[good_mask])[:, 0, 0]
                dy = (next_pts[good_mask] - prev_pts[good_mask])[:, 0, 1]
                magnitudes = np.sqrt(dx**2 + dy**2)
                mean_mag = float(np.mean(magnitudes))
                mean_dir = float(np.arctan2(np.mean(dy), np.mean(dx)))
                _prev_gray = gray_frame.copy()
                _prev_landmarks = landmarks_px
                return {"flow_magnitude": mean_mag, "flow_direction": mean_dir}
    except cv2.error:
        pass

    _prev_gray = gray_frame.copy()
    _prev_landmarks = landmarks_px
    return {"flow_magnitude": 0.0, "flow_direction": 0.0}

# =====================================================================
# CV TECHNIQUE 14: Camera Calibration — Focal Length Estimation
# =====================================================================

def estimate_focal_length(face_width_px, frame_width, assumed_face_width_cm=15.0):
    """[CV: Camera Calibration — Focal Length from Known Object Size]
    Estimates apparent focal length from the detected face width,
    assuming average human face width ~15cm. This is a simplified
    pinhole camera model: f = (face_px * Z) / face_cm.
    Used to estimate approximate face distance for art scale."""
    if face_width_px < 10:
        return {"focal_length_px": 0.0, "estimated_distance_cm": 0.0}
    # Assume face is at ~60cm for a typical webcam session
    assumed_distance_cm = 60.0
    focal_length_px = (face_width_px * assumed_distance_cm) / assumed_face_width_cm
    # Reverse: estimate actual distance
    estimated_distance = (assumed_face_width_cm * focal_length_px) / max(1, face_width_px)
    return {
        "focal_length_px":       float(focal_length_px),
        "estimated_distance_cm": float(estimated_distance),
    }

# =====================================================================
# CV TECHNIQUE 15: Image Segmentation — Skin Color Segmentation
# =====================================================================

def segment_skin(face_rgb):
    """[CV: Image Segmentation — Skin Color Detection in YCrCb]
    Segments skin pixels using YCrCb color space thresholds.
    Returns skin mask and skin ratio. Skin ratio indicates how much
    of the face crop is actual skin vs hair/background.
    Used to validate face crop quality."""
    img = _safe(face_rgb)
    ycrcb = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)
    # Standard skin color range in YCrCb
    lower = np.array([0, 133, 77], dtype=np.uint8)
    upper = np.array([255, 173, 127], dtype=np.uint8)
    skin_mask = cv2.inRange(ycrcb, lower, upper)
    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_CLOSE, kernel)
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_OPEN, kernel)
    skin_ratio = float(np.count_nonzero(skin_mask)) / max(1, skin_mask.size)
    return skin_mask, skin_ratio

# =====================================================================
# CV TECHNIQUE 16: Appearance Matching — Histogram Comparison
# =====================================================================

def compute_appearance_histogram(face_rgb):
    """[CV: Appearance Matching — Color Histogram for Face Re-ID]
    Computes a normalised HSV histogram of the face region.
    Used to match faces across frames when bounding boxes shift.
    Correlation score > 0.7 indicates same person."""
    img = _safe(face_rgb)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    # Compute 2D histogram on H and S channels
    hist = cv2.calcHist([hsv], [0, 1], None, [30, 32], [0, 180, 0, 256])
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist

def match_face_histogram(hist_new, threshold=0.65):
    """[CV: Appearance Matching — Histogram Correlation]
    Compares new face histogram against stored histograms to find
    the best match. Uses Bhattacharyya distance for robustness."""
    global _face_histograms, _next_face_id
    best_id = -1
    best_score = -1.0
    for fid, hist_stored in _face_histograms.items():
        score = cv2.compareHist(hist_new, hist_stored, cv2.HISTCMP_CORREL)
        if score > best_score:
            best_score = score
            best_id = fid
    if best_score >= threshold and best_id >= 0:
        _face_histograms[best_id] = hist_new  # update
        return best_id, best_score
    # New face
    new_id = _next_face_id
    _next_face_id += 1
    _face_histograms[new_id] = hist_new
    return new_id, 0.0

# =====================================================================
# CV TECHNIQUE 17: Object Tracking — Centroid Tracker
# =====================================================================

def track_face_centroid(box, face_id):
    """[CV: Object Tracking — Centroid-Based Face Tracking]
    Maintains a simple centroid tracker to assign consistent IDs
    to faces across frames. Helps the art engine maintain continuity
    for a specific person."""
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    _face_centroids[face_id] = (cx, cy)
    return face_id, (cx, cy)

# =====================================================================
# Face alignment + geometry (existing, with technique labels)
# =====================================================================

def align_face(face_rgb, landmarks_px):
    """[CV: Image Formation — Geometric Transformation / Face Alignment]
    Rotates crop so both eyes lie on a horizontal line using an
    affine warp (projective geometry from Image Formation)."""
    img = _safe(face_rgb)
    lx  = np.mean([landmarks_px[i][0] for i in LEFT_EYE_IDX])
    ly  = np.mean([landmarks_px[i][1] for i in LEFT_EYE_IDX])
    rx  = np.mean([landmarks_px[i][0] for i in RIGHT_EYE_IDX])
    ry  = np.mean([landmarks_px[i][1] for i in RIGHT_EYE_IDX])
    angle = np.degrees(np.arctan2(ry - ly, rx - lx))
    h, w  = img.shape[:2]
    M     = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h),
                          flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REFLECT)

def compute_geometry_features(landmarks_px, h, w):
    """[CV: Face Detection — Geometric Feature Extraction from Landmarks]
    Computes EAR (eye aspect ratio), MAR (mouth aspect ratio),
    brow raise, and smile metrics from 468 facial landmarks."""
    def pt(idx):
        return np.array(landmarks_px[idx], dtype=float)
    def ear(eye_idx):
        p  = [pt(i) for i in eye_idx]
        v1 = np.linalg.norm(p[1] - p[5])
        v2 = np.linalg.norm(p[2] - p[4])
        h_ = np.linalg.norm(p[0] - p[3])
        return (v1 + v2) / (2.0 * h_ + 1e-6)
    avg_ear = (ear(LEFT_EYE_IDX) + ear(RIGHT_EYE_IDX)) / 2.0
    m   = [pt(i) for i in MOUTH_IDX]
    mar = (np.linalg.norm(m[2] - m[3]) + np.linalg.norm(m[4] - m[5])) / \
          (2.0 * np.linalg.norm(m[0] - m[1]) + 1e-6)
    lbrow_y    = np.mean([pt(i)[1] for i in LEFT_BROW_IDX])
    rbrow_y    = np.mean([pt(i)[1] for i in RIGHT_BROW_IDX])
    leye_y     = np.mean([pt(i)[1] for i in LEFT_EYE_IDX])
    reye_y     = np.mean([pt(i)[1] for i in RIGHT_EYE_IDX])
    brow_raise = ((leye_y - lbrow_y) + (reye_y - rbrow_y)) / (2.0 * h + 1e-6)
    smile = (pt(MOUTH_IDX[5])[1] -
             (pt(MOUTH_IDX[0])[1] + pt(MOUTH_IDX[1])[1]) / 2.0) / (h + 1e-6)
    return {"EAR": avg_ear, "MAR": mar, "BrowRaise": brow_raise, "Smile": smile}

def adjust_scores_with_geometry(scores_dict, geo):
    """[CV: Face Detection — Geometry-Based Score Adjustment]
    Uses geometric ratios from landmarks to refine emotion scores."""
    adj = scores_dict.copy()
    if geo["MAR"] > 0.35:
        adj["Surprise"] = adj.get("Surprise", 0) * 1.4
        adj["Happy"]    = adj.get("Happy",    0) * 1.2
    if geo["BrowRaise"] > 0.06:
        adj["Surprise"] = adj.get("Surprise", 0) * 1.3
        adj["Fear"]     = adj.get("Fear",     0) * 1.2
    if geo["EAR"] < 0.22:
        adj["Happy"] = adj.get("Happy", 0) * 0.7
        adj["Sad"]   = adj.get("Sad",   0) * 1.2
        adj["Angry"] = adj.get("Angry", 0) * 1.2
    if geo["Smile"] > 0.02:
        adj["Happy"] = adj.get("Happy", 0) * 1.3
    total = sum(adj.values()) + 1e-9
    return {k: v / total for k, v in adj.items()}

# =====================================================================
# Landmark quality gate
# =====================================================================

def get_landmark_quality(face_rgb):
    """[CV: Face Detection — MediaPipe Landmark Quality Gate]"""
    img      = _safe(face_rgb)
    h, w     = img.shape[:2]
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img)
    result   = face_mesh.detect(mp_image)
    if not result.face_landmarks:
        return None, 0.0
    lms          = result.face_landmarks[0]
    landmarks_px = [(int(lm.x * w), int(lm.y * h)) for lm in lms]
    key_indices  = LEFT_EYE_IDX + RIGHT_EYE_IDX + MOUTH_IDX + [NOSE_TIP_IDX]
    quality      = float(np.mean([lms[i].presence for i in key_indices]))
    return landmarks_px, quality

# =====================================================================
# Master extraction: runs ALL CV techniques on a face crop
# =====================================================================

def extract_all_cv_features(face_rgb, box, frame_gray):
    """Runs every CV technique on a single face crop and returns
    a unified feature dictionary that the art engine consumes."""
    features = {}

    # Technique 3: Binary Images
    try:
        _, face_ratio = compute_binary_face_mask(face_rgb)
        features["face_ratio"] = face_ratio
    except Exception:
        features["face_ratio"] = 0.5

    # Technique 6: Edge Detection
    try:
        edge_feats = compute_edge_features(face_rgb)
        features.update(edge_feats)
    except Exception:
        features["edge_density"] = 0.0
        features["sobel_mag_mean"] = 0.0

    # Technique 7: Boundary Detection
    try:
        if "canny_edges" in features:
            boundary_feats = compute_boundary_features(features["canny_edges"])
        else:
            boundary_feats = {"n_contours": 0, "avg_contour_area": 0, "max_depth": 0}
        features.update(boundary_feats)
    except Exception:
        features["n_contours"] = 0

    # Technique 8: SIFT
    try:
        sift_feats = compute_sift_features(face_rgb)
        features.update(sift_feats)
    except Exception:
        features["sift_count"] = 0

    # Technique 10: Radiometry
    try:
        refl_feats = compute_reflectance_features(face_rgb)
        features.update(refl_feats)
    except Exception:
        features["mean_luminance"] = 0.5

    # Technique 11: Shape from Shading
    try:
        sfs_feats = compute_shape_from_shading(
            features.get("gradient_mag"), features.get("gradient_dir"))
        features.update(sfs_feats)
    except Exception:
        features["curvature"] = 0.0
        features["dominant_angle"] = 0.0

    # Technique 12: Depth from Defocus
    try:
        features["focus_measure"] = compute_focus_measure(face_rgb)
    except Exception:
        features["focus_measure"] = 100.0

    # Technique 14: Camera Calibration
    try:
        x1, y1, x2, y2 = box
        face_width_px = x2 - x1
        cal_feats = estimate_focal_length(face_width_px, frame_gray.shape[1] if frame_gray is not None else 640)
        features.update(cal_feats)
    except Exception:
        features["estimated_distance_cm"] = 60.0

    # Technique 15: Skin Segmentation
    try:
        _, skin_ratio = segment_skin(face_rgb)
        features["skin_ratio"] = skin_ratio
    except Exception:
        features["skin_ratio"] = 0.5

    # Technique 16: Appearance Matching
    try:
        hist = compute_appearance_histogram(face_rgb)
        face_id, match_score = match_face_histogram(hist)
        features["face_id"] = face_id
        features["match_score"] = match_score
    except Exception:
        features["face_id"] = 0
        features["match_score"] = 0.0

    # Clean up non-serializable items
    for key in ["gradient_dir", "gradient_mag", "canny_edges"]:
        features.pop(key, None)

    return features

# =====================================================================
# Master preprocess_face (enhanced with technique labels)
# =====================================================================

def preprocess_face(face_rgb):
    """Full enhancement pipeline with labeled CV techniques:
      1. [Image Processing I]  White balance
      2. [Face Detection]      Landmark quality gate
      3. [Image Formation]     Face alignment
      4. [Image Processing I]  CLAHE
      5. [Image Processing I]  Gamma correction
      6. [Image Processing II] Unsharp mask
    Returns (enhanced_uint8_rgb, quality_score)."""
    try:
        img = _safe(face_rgb)
    except Exception:
        return None, 0.0
    try:
        img = apply_white_balance(img)
    except Exception:
        pass
    landmarks_px, quality = get_landmark_quality(img)
    if landmarks_px is None or quality < QUALITY_THRESHOLD:
        return None, quality
    try:
        img = align_face(img, landmarks_px)
        img = apply_clahe(img)
        img = apply_gamma(img, gamma=1.2)
        img = apply_sharpen(img)
    except Exception as e:
        print(f"preprocess_face partial failure: {e}")
    return img, quality


def preprocess_face_fallback(raw_crop):
    """Lightweight fallback: white balance + CLAHE only."""
    try:
        img = _safe(raw_crop)
        img = apply_white_balance(img)
        img = apply_clahe(img)
        return img
    except Exception:
        if raw_crop is not None:
            return np.ascontiguousarray(raw_crop, dtype=np.uint8)
        raise

print("✅ Advanced CV Pipeline ready — 17 techniques loaded")
print("   Imaging: Image Formation, Sensing, Binary Images, Processing I & II")
print("   Features: Edge Detection, Boundary Detection, SIFT, Face Detection")
print("   Reconstruction I: Radiometry, Shape from Shading, Depth from Defocus")
print("   Reconstruction II: Optical Flow, Camera Calibration")
print("   Perception: Skin Segmentation, Appearance Matching, Object Tracking")



model.safetensors:   0%|          | 0.00/344M [00:00<?, ?B/s]

✅ Advanced CV Pipeline ready — 17 techniques loaded
   Imaging: Image Formation, Sensing, Binary Images, Processing I & II
   Features: Edge Detection, Boundary Detection, SIFT, Face Detection
   Reconstruction I: Radiometry, Shape from Shading, Depth from Defocus
   Reconstruction II: Optical Flow, Camera Calibration
   Perception: Skin Segmentation, Appearance Matching, Object Tracking


In [6]:
# ============================================================
# Cell 5: Training Utilities — Focal Loss + Augmentations
# (Only needed if fine-tuning. Safe to skip for live inference.)
# ============================================================
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

class FocalLoss(nn.Module):
    """Focal Loss — down-weights easy negatives, focuses on hard examples."""
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        log_prob   = F.log_softmax(logits, dim=-1)
        prob       = log_prob.exp()
        log_pt     = log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt         = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_term = (1 - pt) ** self.gamma
        loss       = -focal_term * log_pt
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean() if self.reduction == "mean" else loss.sum()

_class_counts   = torch.tensor([4953., 547., 5121., 8989., 6077., 4002., 6198.])
_alpha          = (_class_counts ** -1)
_alpha          = _alpha / _alpha.sum()
focal_criterion = FocalLoss(gamma=2.0, alpha=_alpha.to(device))

def get_training_transforms(image_size=224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=12),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
        transforms.ColorJitter(brightness=0.4, contrast=0.4,
                               saturation=0.3, hue=0.05),
        transforms.RandomGrayscale(p=0.15),
        transforms.RandomEqualize(p=0.2),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.10),
                                 ratio=(0.5, 2.0), value=0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

def get_inference_transforms(image_size=224):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

print("✅ Training utilities ready")


✅ Training utilities ready


In [7]:
# ============================================================
# Cell 6: Drawing Helpers
# ============================================================

def draw_emotion_bar(frame, all_results):
    """Colour-coded emotion score bars along the bottom."""
    h, w       = frame.shape[:2]
    bar_h, pad = 60, 6
    n  = len(all_results)
    bw = (w - pad * (n + 1)) // n
    cv2.rectangle(frame, (0, h - bar_h - 20), (w, h), (30, 30, 30), -1)
    for i, r in enumerate(all_results):
        name  = EMOTION_LABELS.get(r["label"], r.get("name", r["label"]))
        score = r["score"]
        color = EMOTION_COLORS.get(name, (200, 200, 200))
        x0, yt, yb = pad + i * (bw + pad), h - bar_h - 5, h - 20
        cv2.rectangle(frame, (x0, yt),      (x0 + bw, yb), (60, 60, 60), -1)
        fh = int((yb - yt) * score)
        cv2.rectangle(frame, (x0, yb - fh), (x0 + bw, yb), color, -1)
        sz = cv2.getTextSize(name, cv2.FONT_HERSHEY_SIMPLEX, 0.35, 1)[0]
        cv2.putText(frame, name, (x0 + (bw - sz[0]) // 2, h - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, (220, 220, 220), 1)
        pct = f"{score * 100:.0f}%"
        ps  = cv2.getTextSize(pct, cv2.FONT_HERSHEY_SIMPLEX, 0.3, 1)[0]
        cv2.putText(frame, pct, (x0 + (bw - ps[0]) // 2, yb - fh - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.3, (255, 255, 255), 1)
    return frame


def draw_race_bar(frame, race_scores):
    """Colour-coded race/ethnicity score bars along the top."""
    h, w       = frame.shape[:2]
    bar_h, pad = 50, 6
    races = sorted(race_scores.items(), key=lambda x: x[0])
    n  = len(races)
    bw = (w - pad * (n + 1)) // n
    cv2.rectangle(frame, (0, 0), (w, bar_h + 18), (20, 20, 20), -1)
    for i, (race, score_pct) in enumerate(races):
        color = RACE_COLORS.get(race.lower(), (200, 200, 200))
        x0, yt, yb = pad + i * (bw + pad), 5, bar_h
        cv2.rectangle(frame, (x0, yt),      (x0 + bw, yb), (60, 60, 60), -1)
        fh = int((yb - yt) * (score_pct / 100.0))
        cv2.rectangle(frame, (x0, yb - fh), (x0 + bw, yb), color, -1)
        short = race.title()
        sz = cv2.getTextSize(short, cv2.FONT_HERSHEY_SIMPLEX, 0.30, 1)[0]
        cv2.putText(frame, short, (x0 + (bw - sz[0]) // 2, yb + 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.30, (200, 200, 200), 1)
        if fh > 10:
            pct = f"{score_pct:.0f}%"
            ps  = cv2.getTextSize(pct, cv2.FONT_HERSHEY_SIMPLEX, 0.28, 1)[0]
            cv2.putText(frame, pct, (x0 + (bw - ps[0]) // 2, yb - fh - 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.28, (255, 255, 255), 1)
    return frame


def frame_to_jpeg(frame):
    _, buf = cv2.imencode('.jpg', frame)
    return buf.tobytes()

print("✅ Drawing helpers ready")


✅ Drawing helpers ready


In [9]:
# ============================================================
# Cell 7: Live Inference Loop
#
# STOP BUTTON: HTML button sets window._stopLoop = true.
#   capture_frame_from_browser() checks this flag each frame
#   and returns None — breaking the loop cleanly.
#
# CAMERA: Always re-initialised here because Colab resets the
#   JS context (window.*) between cell executions.
# ============================================================

# ── Always init camera fresh — JS context resets between cells ───
print("📷 Starting camera...")
print("   → Click Allow if a permission dialog appears.")
_result = _init_camera()
if _result != 'OK':
    raise RuntimeError(f"Camera failed: {_result}. Re-run this cell.")
print("✅ Camera ready.")

# ── Reset smoothing buffers ───────────────────────────────────────
emotion_history_per_face.clear()
race_history_per_face.clear()

# ── UI ────────────────────────────────────────────────────────────
img_widget = widgets.Image(format='jpeg', width=640)
status_lbl = widgets.Label(value="🎥 Running — click Stop to end.")
display(widgets.VBox([status_lbl, img_widget]))
inject_stop_button()

print("🎥 Starting live feed...")

# ── Main loop ─────────────────────────────────────────────────────
while True:
    try:
        frame = capture_frame_from_browser()
        if frame is None:
            status_lbl.value = "⏹ Stopped."
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # ── Face detection ────────────────────────────────────
        boxes, _ = mtcnn.detect(frame_rgb)
        if boxes is None:
            img_widget.value = frame_to_jpeg(frame)
            continue

        all_results   = None
        dominant_race = ""
        race_scores   = {}
        quality       = 0.0

        for box in boxes:
            x1, y1, x2, y2 = [max(0, int(b)) for b in box]
            raw_crop = frame_rgb[y1:y2, x1:x2]
            if raw_crop is None or raw_crop.size == 0:
                continue

            # ── Enhancement pipeline ──────────────────────────
            try:
                processed_crop, quality = preprocess_face(raw_crop)
                inference_crop = processed_crop if processed_crop is not None \
                                 else preprocess_face_fallback(raw_crop)
            except Exception as pe:
                status_lbl.value = f"Preprocess warn: {pe}"
                inference_crop = np.ascontiguousarray(raw_crop, dtype=np.uint8)
                quality = 0.0

            pil_crop = Image.fromarray(inference_crop)

            # ── Emotion inference ─────────────────────────────
            try:
                all_results = emotion_model(pil_crop, top_k=7)
                all_results = sorted(all_results, key=lambda r: r["label"])
                top         = max(all_results, key=lambda r: r["score"])
                top_score   = top["score"]
                _fid = 0  # cell 7 has no CV feats; use single ID
                top_emotion = smooth_emotion(
                    EMOTION_LABELS.get(top["label"], top["label"]), face_id=_fid
                )
            except Exception as ee:
                status_lbl.value = f"Emotion error: {ee}"
                continue

            # ── Race inference ────────────────────────────────
            try:
                race_results = race_model(pil_crop, top_k=6)
                race_scores  = {r["label"].lower(): r["score"] * 100
                                for r in race_results}
                top_race     = max(race_results, key=lambda r: r["score"])
                if top_race["score"] >= RACE_CONF_THRESHOLD:
                    dominant_race = smooth_race(top_race["label"].title(), face_id=fid if "fid" in dir() else current_cv_feats.get("face_id", 0))
            except Exception:
                pass

            # ── Draw bounding box + label ─────────────────────
            box_color = (0, 255, 0) if quality >= QUALITY_THRESHOLD \
                        else (140, 140, 140)
            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
            if top_score >= EMOTION_CONF_THRESHOLD:
                race_tag  = f" | {dominant_race}" if dominant_race else ""
                cv2.putText(frame,
                            f"{top_emotion} ({top_score:.2f}){race_tag}",
                            (x1, max(y1 - 10, 70)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.75, box_color, 2)
            else:
                cv2.putText(frame, "low confidence",
                            (x1, max(y1 - 10, 70)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (120, 120, 120), 1)

        # ── Draw bars ─────────────────────────────────────────
        if all_results:
            frame = draw_emotion_bar(frame, all_results)
        if race_scores:
            frame = draw_race_bar(frame, race_scores)

        img_widget.value = frame_to_jpeg(frame)
        status_lbl.value = f"🎥 Live | {dominant_race or '—'} | Q:{quality:.0%}"

    except Exception as e:
        status_lbl.value = f"❌ Error: {e}"
        break

print("✅ Done.")



📷 Starting camera...
   → Click Allow if a permission dialog appears.
✅ Camera ready.


🎥 Starting live feed...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ Done.


In [10]:
# ============================================================
# Cell 8A: Cultural-Emotional Color Matrix
#
# Maps (emotion, race) → (primary_color, accent_color, bg_color)
# Colors are chosen based on documented cultural symbolism:
#   - White: purity/joy in Western cultures → mourning in East Asian
#   - Red:   luck/celebration in Asian → danger/anger in Western
#   - Gold:  prosperity across Asian cultures
#   - Blue:  sadness (universal) but tone varies by culture
#
# Each entry: (primary_rgb, accent_rgb, bg_rgb)
# Colors are in RGB for PIL compatibility.
# ============================================================

# ── Cultural palette definitions ─────────────────────────────
# Format: CULTURAL_MATRIX[(emotion, race)] = (primary, accent, bg)
# 'race' keys must match lowercased output of cledoux42/Ethnicity_Test_v003
# Known labels: 'asian', 'black' / 'african', 'caucasian' / 'white',
#               'hispanic' / 'latino hispanic', 'indian', 'middle eastern'

CULTURAL_MATRIX = {
    # ── Happy ────────────────────────────────────────────────────
    ("Happy", "asian"):          ((255, 200, 0),   (255, 80,  0),   (255, 245, 180)),  # Gold + red-orange: luck, celebration
    ("Happy", "african"):        ((0,   180, 100), (255, 200, 0),   (220, 255, 235)),  # Green + gold: vitality, abundance
    ("Happy", "black"):          ((0,   180, 100), (255, 200, 0),   (220, 255, 235)),
    ("Happy", "caucasian"):      ((255, 230, 80),  (100, 200, 255), (255, 255, 220)),  # Sunny yellow + sky blue
    ("Happy", "white"):          ((255, 230, 80),  (100, 200, 255), (255, 255, 220)),
    ("Happy", "hispanic"):       ((255, 80,  0),   (255, 210, 0),   (255, 235, 200)),  # Fiesta red + marigold
    ("Happy", "latino hispanic"): ((255, 80, 0),   (255, 210, 0),   (255, 235, 200)),
    ("Happy", "indian"):         ((255, 140, 0),   (200, 0,   80),  (255, 240, 200)),  # Saffron + deep pink: Holi
    ("Happy", "middle eastern"): ((0,   160, 120), (200, 160, 0),   (220, 245, 235)),  # Teal + gold: joy, prosperity

    # ── Sad ─────────────────────────────────────────────────────
    ("Sad", "asian"):            ((80,  80,  160), (180, 180, 220), (220, 220, 245)),  # Muted indigo: grief (avoids white=mourning)
    ("Sad", "african"):          ((60,  60,  120), (120, 120, 180), (210, 210, 235)),  # Deep violet
    ("Sad", "black"):            ((60,  60,  120), (120, 120, 180), (210, 210, 235)),
    ("Sad", "caucasian"):        ((70,  100, 180), (150, 170, 220), (215, 225, 245)),  # Classic blue-grey
    ("Sad", "white"):            ((70,  100, 180), (150, 170, 220), (215, 225, 245)),
    ("Sad", "hispanic"):         ((80,  80,  160), (160, 140, 200), (220, 215, 240)),  # Purple-indigo: luto
    ("Sad", "latino hispanic"):  ((80,  80,  160), (160, 140, 200), (220, 215, 240)),
    ("Sad", "indian"):           ((50,  50,  100), (100, 80,  160), (210, 205, 235)),  # Deep blue: shoka
    ("Sad", "middle eastern"):   ((90,  90,  150), (160, 150, 200), (220, 218, 240)),  # Indigo

    # ── Angry ────────────────────────────────────────────────────
    ("Angry", "asian"):          ((180, 0,   0),   (255, 80,  0),   (255, 200, 180)),  # Dark red + orange
    ("Angry", "african"):        ((200, 20,  0),   (255, 60,  0),   (255, 210, 190)),
    ("Angry", "black"):          ((200, 20,  0),   (255, 60,  0),   (255, 210, 190)),
    ("Angry", "caucasian"):      ((220, 30,  30),  (255, 100, 0),   (255, 215, 200)),
    ("Angry", "white"):          ((220, 30,  30),  (255, 100, 0),   (255, 215, 200)),
    ("Angry", "hispanic"):       ((200, 0,   0),   (255, 50,  0),   (255, 205, 185)),  # Intense red
    ("Angry", "latino hispanic"): ((200, 0,  0),   (255, 50,  0),   (255, 205, 185)),
    ("Angry", "indian"):         ((180, 0,   20),  (255, 40,  0),   (255, 205, 190)),
    ("Angry", "middle eastern"): ((160, 0,   0),   (220, 60,  0),   (255, 205, 185)),

    # ── Fear ─────────────────────────────────────────────────────
    ("Fear", "asian"):           ((80,  0,  120), (140, 60,  180), (225, 200, 240)),
    ("Fear", "african"):         ((60,  0,  100), (120, 40,  160), (220, 195, 238)),
    ("Fear", "black"):           ((60,  0,  100), (120, 40,  160), (220, 195, 238)),
    ("Fear", "caucasian"):       ((90,  90,  90), (160, 160, 160), (225, 225, 225)),  # Drained grey
    ("Fear", "white"):           ((90,  90,  90), (160, 160, 160), (225, 225, 225)),
    ("Fear", "hispanic"):        ((70,  0,  110), (130, 50,  170), (222, 198, 240)),
    ("Fear", "latino hispanic"): ((70,  0,  110), (130, 50,  170), (222, 198, 240)),
    ("Fear", "indian"):          ((60,  0,   80), (110, 30,  150), (220, 192, 236)),
    ("Fear", "middle eastern"): ((80,  0,  100), (130, 40,  160), (222, 196, 238)),

    # ── Surprise ─────────────────────────────────────────────────
    ("Surprise", "asian"):        ((255, 200, 0),  (0,  200, 200), (240, 250, 220)),
    ("Surprise", "african"):      ((255, 140, 0),  (0,  180, 180), (240, 245, 215)),
    ("Surprise", "black"):        ((255, 140, 0),  (0,  180, 180), (240, 245, 215)),
    ("Surprise", "caucasian"):    ((0,  180, 220), (255, 200, 0),  (220, 245, 255)),
    ("Surprise", "white"):        ((0,  180, 220), (255, 200, 0),  (220, 245, 255)),
    ("Surprise", "hispanic"):     ((255, 100, 0),  (0,  200, 150), (240, 245, 215)),
    ("Surprise", "latino hispanic"): ((255, 100, 0), (0, 200, 150), (240, 245, 215)),
    ("Surprise", "indian"):       ((255, 140, 0),  (200, 0, 100),  (255, 235, 210)),
    ("Surprise", "middle eastern"): ((0, 160, 200), (255, 180, 0), (220, 242, 255)),

    # ── Disgust ──────────────────────────────────────────────────
    ("Disgust", "asian"):         ((60,  120, 40), (140, 160, 80), (210, 230, 200)),
    ("Disgust", "african"):       ((50,  110, 30), (120, 150, 70), (205, 228, 195)),
    ("Disgust", "black"):         ((50,  110, 30), (120, 150, 70), (205, 228, 195)),
    ("Disgust", "caucasian"):     ((80,  130, 50), (150, 170, 90), (215, 232, 205)),
    ("Disgust", "white"):         ((80,  130, 50), (150, 170, 90), (215, 232, 205)),
    ("Disgust", "hispanic"):      ((70,  120, 40), (140, 160, 80), (210, 230, 200)),
    ("Disgust", "latino hispanic"): ((70, 120, 40), (140, 160, 80), (210, 230, 200)),
    ("Disgust", "indian"):        ((60,  110, 40), (130, 155, 75), (208, 228, 198)),
    ("Disgust", "middle eastern"): ((65, 115, 45), (135, 158, 78), (209, 229, 199)),

    # ── Neutral ──────────────────────────────────────────────────
    ("Neutral", "asian"):         ((160, 160, 180), (200, 200, 220), (235, 235, 245)),
    ("Neutral", "african"):       ((150, 150, 160), (190, 190, 210), (232, 232, 242)),
    ("Neutral", "black"):         ((150, 150, 160), (190, 190, 210), (232, 232, 242)),
    ("Neutral", "caucasian"):     ((170, 170, 185), (210, 210, 225), (238, 238, 248)),
    ("Neutral", "white"):         ((170, 170, 185), (210, 210, 225), (238, 238, 248)),
    ("Neutral", "hispanic"):      ((160, 155, 170), (200, 195, 215), (235, 232, 245)),
    ("Neutral", "latino hispanic"): ((160, 155, 170), (200, 195, 215), (235, 232, 245)),
    ("Neutral", "indian"):        ((155, 150, 170), (195, 190, 215), (233, 230, 244)),
    ("Neutral", "middle eastern"): ((158, 153, 173), (198, 193, 217), (234, 231, 245)),
}

# ── Fallback palette ─────────────────────────────────────────
# Used when (emotion, race) key not in matrix
EMOTION_FALLBACK_PALETTE = {
    "Happy":   ((255, 220, 60),  (255, 120, 0),   (255, 250, 200)),
    "Sad":     ((70,  100, 180), (150, 170, 220),  (215, 225, 245)),
    "Angry":   ((210, 30,  30),  (255, 100, 0),    (255, 215, 200)),
    "Fear":    ((90,  40,  140), (160, 100, 200),  (230, 210, 245)),
    "Surprise":((0,   180, 220), (255, 200, 60),   (220, 248, 255)),
    "Disgust": ((70,  130, 50),  (150, 170, 90),   (215, 232, 205)),
    "Neutral": ((160, 160, 175), (205, 205, 220),  (235, 235, 245)),
}

def get_palette(emotion, race):
    """
    Returns (primary_rgb, accent_rgb, bg_rgb) for the given
    emotion and race. Falls back gracefully when the key is absent.
    """
    race_key = race.lower().strip() if race else ""
    key = (emotion, race_key)
    if key in CULTURAL_MATRIX:
        return CULTURAL_MATRIX[key]
    # Try partial match on race key (e.g. 'african american' -> 'african')
    for stored_emotion, stored_race in CULTURAL_MATRIX:
        if stored_emotion == emotion and stored_race in race_key:
            return CULTURAL_MATRIX[(stored_emotion, stored_race)]
    return EMOTION_FALLBACK_PALETTE.get(
        emotion,
        ((180, 180, 180), (220, 220, 220), (245, 245, 245))
    )

print("✅ Cultural-Emotional Color Matrix ready")
print(f"   {len(CULTURAL_MATRIX)} culturally-mapped (emotion, race) palettes loaded")


✅ Cultural-Emotional Color Matrix ready
   63 culturally-mapped (emotion, race) palettes loaded


In [11]:
# ============================================================
# Cell 8B: Immersive Therapeutic Art Engine v13
#          Race-Differentiated Patterns + CV-Driven Art
#
# ┌──────────────────────────────────────────────────────────┐
# │  KEY UPGRADE: Each race produces FUNDAMENTALLY DIFFERENT │
# │  art — not just repositioned patterns. Each race gets:   │
# │   • Different SHADES of the same base color family       │
# │   • Completely different PATTERN GENERATORS               │
# │   • Different GEOMETRIC PRIMITIVES and compositions      │
# │                                                          │
# │  CV FEATURES DRIVING ART:                                │
# │   • edge_density → pattern complexity                    │
# │   • sift_count → detail level                            │
# │   • flow_magnitude → animation speed                     │
# │   • focus_measure → art clarity/blur                     │
# │   • curvature → shape curvature in art                   │
# │   • dominant_angle → lighting direction in art           │
# │   • estimated_distance → art scale                       │
# │   • skin_ratio → art opacity confidence                  │
# └──────────────────────────────────────────────────────────┘
#
# THERAPEUTIC RULES (from requirements):
#   STRESSED (Sad/Angry/Fear/Disgust):
#     → Radial Symmetry / Mandalas / Slow Fractals
#     → Curvilinear lines, soft rounded edges
#     → Cool Analogous colors (blues, teals, greens)
#     → Slow breathing tempo (6 cycles/min)
#
#   HAPPY (Happy/Surprise):
#     → Expansive Flowing / Neurographic patterns
#     → Dynamic asymmetry, upward lines
#     → Warm High-Contrast Saturation (golds, magentas)
#     → Reactive playful motion
#
#   NEUTRAL:
#     → Gentle kinetic waves, balanced palette
# ============================================================
import math, random
from PIL import Image, ImageDraw, ImageFilter
import io

ART_SIZE  = 512
art_phase = 0

# =====================================================================
# COLOR SYSTEM: Different SHADES per race within the same hue family
# =====================================================================

def _hue_shift(base_rgb, hue_offset, sat_mult=1.0, val_mult=1.0):
    """Shift a color's hue in HSV space to create race-specific shades."""
    r, g, b = base_rgb[0]/255.0, base_rgb[1]/255.0, base_rgb[2]/255.0
    mx, mn = max(r, g, b), min(r, g, b)
    diff = mx - mn
    # RGB to HSV
    if diff < 1e-6:
        h = 0
    elif mx == r:
        h = (60 * ((g - b) / diff) + 360) % 360
    elif mx == g:
        h = 60 * ((b - r) / diff) + 120
    else:
        h = 60 * ((r - g) / diff) + 240
    s = 0 if mx < 1e-6 else diff / mx
    v = mx
    # Apply shifts
    h = (h + hue_offset) % 360
    s = min(1.0, max(0.0, s * sat_mult))
    v = min(1.0, max(0.0, v * val_mult))
    # HSV to RGB
    c = v * s
    x = c * (1 - abs((h / 60) % 2 - 1))
    m = v - c
    if h < 60:    r1, g1, b1 = c, x, 0
    elif h < 120: r1, g1, b1 = x, c, 0
    elif h < 180: r1, g1, b1 = 0, c, x
    elif h < 240: r1, g1, b1 = 0, x, c
    elif h < 300: r1, g1, b1 = x, 0, c
    else:         r1, g1, b1 = c, 0, x
    return (int((r1+m)*255), int((g1+m)*255), int((b1+m)*255))

# Race-specific shade modifiers: (hue_offset, sat_multiplier, val_multiplier)
RACE_SHADE_MAP = {
    "asian":          (15,  0.85, 1.05),   # warmer, slightly desaturated, brighter
    "african":        (-10, 1.20, 0.85),   # cooler shift, richer saturation, deeper
    "black":          (-10, 1.20, 0.85),
    "caucasian":      (0,   0.75, 1.10),   # neutral hue, softer saturation, lighter
    "white":          (0,   0.75, 1.10),
    "hispanic":       (25,  1.10, 0.95),   # warm shift, vibrant, medium depth
    "latino hispanic":(25,  1.10, 0.95),
    "indian":         (35,  1.15, 0.90),   # strong warm shift, rich, deep
    "middle eastern": (-5,  0.90, 1.00),   # slight cool, elegant saturation
}

def get_race_shaded_palette(base_primary, base_accent, base_bg, race):
    """Create race-specific shades of the same color family."""
    race_key = race.lower().strip() if race else ""
    h_off, s_mult, v_mult = RACE_SHADE_MAP.get(race_key, (0, 1.0, 1.0))
    return (
        _hue_shift(base_primary, h_off, s_mult, v_mult),
        _hue_shift(base_accent,  h_off, s_mult * 0.9, v_mult * 1.05),
        _hue_shift(base_bg,      h_off * 0.3, 1.0, v_mult),
    )

# =====================================================================
# THERAPEUTIC BASE PALETTES (emotion → base colors before race shading)
# =====================================================================

# Stressed emotions → cool analogous (blues, teals, greens)
# Happy emotions → warm high-contrast (golds, magentas)
THERAPEUTIC_TARGETS = {
    # emotion: (intention, bg_dark, bg_light, primary, accent, sparkle)
    "Sad":      ("soothe",   (10, 15, 45),  (80, 140, 200), (60, 130, 190), (40, 100, 170), (150, 200, 240)),
    "Angry":    ("soothe",   (5,  20, 50),  (50, 150, 180), (30, 120, 160), (20, 100, 140), (140, 210, 230)),
    "Fear":     ("soothe",   (10, 25, 40),  (70, 160, 150), (50, 130, 130), (40, 110, 120), (160, 220, 210)),
    "Disgust":  ("soothe",   (10, 30, 35),  (60, 170, 140), (40, 140, 120), (30, 120, 110), (150, 220, 200)),
    "Happy":    ("amplify",  (40, 15, 10),  (255, 200, 50), (255, 140, 30), (240, 60, 120),  (255, 240, 150)),
    "Surprise": ("amplify",  (35, 10, 30),  (240, 160, 60), (220, 100, 180),(200, 60, 200),  (255, 220, 180)),
    "Neutral":  ("balance",  (15, 20, 35),  (120, 160, 200),(90, 130, 180), (70, 150, 170),  (180, 210, 240)),
}

def _c(rgb, a=255):
    return (int(rgb[0]), int(rgb[1]), int(rgb[2]), int(a))

def _lerp(c1, c2, t):
    return tuple(int(c1[i] + (c2[i]-c1[i]) * t) for i in range(3))

# =====================================================================
# LAYER 1: Background — radial gradient (soothe) or diagonal (amplify)
# =====================================================================

def _draw_bg(img, bg_dark, bg_light, phase, intention):
    """Background gradient adapted to therapeutic intention."""
    arr = img.load()
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    max_dist = math.sqrt(cx**2 + cy**2)
    ph = (phase * 0.05) % (2 * math.pi)

    if intention == "soothe":
        # [Stressed] Radial gradient — draws eye inward, containment
        for y in range(ART_SIZE):
            for x in range(ART_SIZE):
                dist = math.sqrt((x - cx)**2 + (y - cy)**2) / max_dist
                t = dist + 0.08 * math.sin(ph + dist * math.pi * 2)
                t = max(0.0, min(1.0, t))
                col = _lerp(bg_light, bg_dark, t)
                arr[x, y] = (col[0], col[1], col[2], 255)
    elif intention == "amplify":
        # [Happy] Diagonal sweep — dynamic, expansive energy
        for y in range(ART_SIZE):
            for x in range(ART_SIZE):
                t = ((x + y) / (2 * ART_SIZE) + 0.1 * math.sin(ph + (x-y)*0.01)) % 1.0
                col = _lerp(bg_dark, bg_light, t)
                arr[x, y] = (col[0], col[1], col[2], 255)
    else:
        # [Neutral] Horizontal gradient
        for y in range(ART_SIZE):
            t = (y / ART_SIZE + 0.06 * math.sin(ph + y * 0.02)) % 1.0
            col = _lerp(bg_dark, bg_light, t)
            for x in range(ART_SIZE):
                arr[x, y] = (col[0], col[1], col[2], 255)

# =====================================================================
# LAYER 2: STRESSED PATTERNS — Radial Symmetry / Mandalas / Fractals
#   Curvilinear, soft, rounded, predictable, hierarchical
#   6 completely different generators per race
# =====================================================================

def _soothe_asian(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Asian — Soothe] Enso circles + flowing water ripples.
    Inspired by Zen garden raking patterns. Concentric incomplete
    circles (Enso) with gentle wave interference."""
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    ph = phase * 0.06  # slow breathing tempo
    n_enso = int(5 + 8 * score)
    edge_mod = min(2.0, 1.0 + cv_feats.get("edge_density", 0) * 5)

    for i in range(n_enso):
        t = i / max(1, n_enso)
        r = int(ART_SIZE * (0.05 + t * 0.42))
        # Incomplete circle (Enso) — gap rotates with phase
        start_angle = int((ph * 30 + i * 40) % 360)
        span = int(280 + 40 * math.sin(ph + i))  # ~280-320 degree arc
        col = _lerp(primary, accent, t)
        lw = max(2, int((7 - t * 4) * score * edge_mod))
        draw.arc([cx-r, cy-r, cx+r, cy+r],
                 start=start_angle, end=start_angle + span,
                 fill=_c(col, int(160 + 60 * (1-t))), width=lw)

    # Water ripple sine waves
    n_waves = int(4 + 6 * score)
    for w in range(n_waves):
        pts = []
        y_base = int(ART_SIZE * (0.2 + 0.6 * w / max(1, n_waves)))
        for x in range(0, ART_SIZE, 3):
            y = y_base + int(15 * score * math.sin(x * 0.03 + ph + w * 0.5))
            pts.append((x, y))
        if len(pts) > 1:
            col = _lerp(accent, sparkle, w / max(1, n_waves))
            draw.line(pts, fill=_c(col, int(80 * score)), width=max(1, int(2 * score)))

def _soothe_african(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[African — Soothe] Adinkra-inspired concentric symbols.
    Layered circular medallions with internal symmetry lines.
    Uses thick curvilinear strokes in rhythmic repetition."""
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    ph = phase * 0.06
    n_layers = int(4 + 6 * score)
    sift_mod = min(2.0, 1.0 + cv_feats.get("sift_count", 0) / 100.0)

    for layer in range(n_layers):
        t = layer / max(1, n_layers)
        r = int(ART_SIZE * (0.08 + t * 0.38))
        col = _lerp(primary, accent, t)
        lw = max(3, int((10 - t * 6) * score))
        # Full concentric circle
        draw.ellipse([cx-r, cy-r, cx+r, cy+r], outline=_c(col, 200), width=lw)
        # Internal symmetry: 4-fold cross inside each ring
        n_sym = int(4 * sift_mod)
        for s in range(n_sym):
            angle = 2 * math.pi * s / n_sym + ph * 0.3
            x1 = cx + int(r * 0.3 * math.cos(angle))
            y1 = cy + int(r * 0.3 * math.sin(angle))
            x2 = cx + int(r * 0.95 * math.cos(angle))
            y2 = cy + int(r * 0.95 * math.sin(angle))
            draw.line([(x1,y1),(x2,y2)], fill=_c(col, 140), width=max(1, int(lw * 0.5)))

    # Dot border ring
    n_dots = int(20 + 30 * score)
    dot_r = int(ART_SIZE * 0.44)
    for d in range(n_dots):
        angle = 2 * math.pi * d / n_dots + ph * 0.2
        dx = cx + int(dot_r * math.cos(angle))
        dy = cy + int(dot_r * math.sin(angle))
        sz = max(2, int(4 * score))
        draw.ellipse([dx-sz, dy-sz, dx+sz, dy+sz], fill=_c(accent, 180))

def _soothe_caucasian(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Caucasian — Soothe] Fibonacci spiral mandala.
    Soft logarithmic spirals emanating from center with petal-like
    curves at golden angle intervals."""
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    ph = phase * 0.06
    golden_angle = 137.508  # degrees
    n_petals = int(8 + 15 * score)
    focus_mod = min(1.5, max(0.5, cv_feats.get("focus_measure", 100) / 200.0))

    for p in range(n_petals):
        angle_deg = p * golden_angle + ph * 20
        angle = math.radians(angle_deg)
        t = p / max(1, n_petals)
        # Logarithmic spiral radius
        r = int(ART_SIZE * 0.03 * math.exp(0.15 * p * focus_mod))
        r = min(r, int(ART_SIZE * 0.46))
        px = cx + int(r * math.cos(angle))
        py = cy + int(r * math.sin(angle))
        # Petal: small ellipse at each point
        petal_size = max(3, int((12 - t * 8) * score))
        col = _lerp(primary, accent, t)
        draw.ellipse([px-petal_size, py-petal_size, px+petal_size, py+petal_size],
                     fill=_c(col, int(140 + 80 * (1-t))), outline=_c(accent, 100), width=1)

    # Central mandala rings
    for ring in range(3):
        r = int(ART_SIZE * (0.03 + ring * 0.04))
        draw.ellipse([cx-r, cy-r, cx+r, cy+r],
                     outline=_c(sparkle, 200), width=max(1, int(3 * score)))

def _soothe_hispanic(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Hispanic — Soothe] Ojo de Dios (God's Eye) diamond mandala.
    Nested rotated squares forming a radiating diamond pattern,
    inspired by Huichol yarn art."""
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    ph = phase * 0.06
    n_layers = int(6 + 10 * score)
    curv_mod = min(1.5, 1.0 + cv_feats.get("curvature", 0) / 20.0)

    for layer in range(n_layers):
        t = layer / max(1, n_layers)
        size = int(ART_SIZE * (0.03 + t * 0.44))
        # Rotate each layer slightly — creates diamond nesting
        rot = ph * 0.4 + layer * math.pi / (8 * curv_mod)
        col = _lerp(primary, accent, t)
        lw = max(2, int((8 - t * 5) * score))
        # Draw rotated diamond (4 corners)
        pts = []
        for corner in range(4):
            a = rot + corner * math.pi / 2
            pts.append((cx + int(size * math.cos(a)),
                        cy + int(size * math.sin(a))))
        draw.polygon(pts, outline=_c(col, int(200 - t * 60)), width=lw)

    # Yarn thread lines radiating from center
    n_threads = int(8 + 12 * score)
    for th in range(n_threads):
        angle = 2 * math.pi * th / n_threads + ph * 0.15
        length = int(ART_SIZE * 0.42 * score)
        ex = cx + int(length * math.cos(angle))
        ey = cy + int(length * math.sin(angle))
        draw.line([(cx,cy),(ex,ey)], fill=_c(sparkle, int(60 * score)), width=1)

def _soothe_indian(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Indian — Soothe] Rangoli-inspired petal mandala.
    Symmetrical flower patterns with dotted outlines, inspired by
    Kolam / Rangoli floor art traditions."""
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    ph = phase * 0.06
    n_petals = int(8 + 8 * score)
    lum_mod = min(1.3, max(0.7, cv_feats.get("mean_luminance", 0.5) * 2))

    # Petal layers
    for ring in range(3):
        r_base = int(ART_SIZE * (0.10 + ring * 0.12))
        for p in range(n_petals):
            angle = 2 * math.pi * p / n_petals + ph * 0.3 + ring * 0.1
            px = cx + int(r_base * math.cos(angle))
            py = cy + int(r_base * math.sin(angle))
            # Tear-drop petal shape (ellipse elongated radially)
            petal_w = max(4, int((18 - ring * 4) * score * lum_mod))
            petal_h = max(6, int((28 - ring * 6) * score * lum_mod))
            col = _lerp(primary, accent, ring / 3)
            bbox = [px - petal_w, py - petal_h, px + petal_w, py + petal_h]
            draw.ellipse(bbox, fill=_c(col, int(150 - ring * 30)),
                         outline=_c(accent, 180), width=1)

    # Dot grid around petals (Kolam dots)
    n_dots = int(16 + 20 * score)
    for d in range(n_dots):
        angle = 2 * math.pi * d / n_dots
        for ring_d in range(2, 5):
            dr = int(ART_SIZE * 0.08 * ring_d)
            dx = cx + int(dr * math.cos(angle + ph * 0.1))
            dy = cy + int(dr * math.sin(angle + ph * 0.1))
            sz = max(1, int(3 * score))
            draw.ellipse([dx-sz, dy-sz, dx+sz, dy+sz], fill=_c(sparkle, 160))

def _soothe_middle_eastern(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Middle Eastern — Soothe] Geometric arabesque tessellation.
    Interlocking star-and-cross pattern inspired by Islamic geometric
    art. Highly ordered, hierarchical — ideal for stress relief."""
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    ph = phase * 0.06
    dist_mod = min(1.3, max(0.7, cv_feats.get("estimated_distance_cm", 60) / 60.0))

    # Generate tessellated 8-point stars on a grid
    grid_size = int(3 + 4 * score)
    cell_size = ART_SIZE // max(1, grid_size)
    for gy in range(grid_size):
        for gx in range(grid_size):
            scx = int(cell_size * (gx + 0.5))
            scy = int(cell_size * (gy + 0.5))
            star_r = int(cell_size * 0.4 * dist_mod)
            # 8-point star: two overlapping squares rotated 45°
            col = _lerp(primary, accent, ((gx + gy) % 3) / 3.0)
            for rot_offset in [0, math.pi / 4]:
                pts = []
                for corner in range(4):
                    a = rot_offset + corner * math.pi / 2 + ph * 0.2
                    pts.append((scx + int(star_r * math.cos(a)),
                                scy + int(star_r * math.sin(a))))
                draw.polygon(pts, outline=_c(col, int(180 * score)), width=max(1, int(3 * score)))
            # Center dot
            draw.ellipse([scx-2, scy-2, scx+2, scy+2], fill=_c(sparkle, int(150 * score)))

# =====================================================================
# LAYER 2: HAPPY PATTERNS — Expansive Flowing / Neurographic
#   Dynamic asymmetry, upward lines, warm high-contrast
#   6 completely different generators per race
# =====================================================================

def _amplify_asian(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Asian — Amplify] Cherry blossom burst — upward floating petals
    with branching curves radiating outward. Neurographic-style
    flowing intersections."""
    cx, cy = ART_SIZE // 2, int(ART_SIZE * 0.6)  # origin below center → upward energy
    ph = phase * 0.15
    n_branches = int(5 + 8 * score)
    flow_speed = 1.0 + cv_feats.get("flow_magnitude", 0) * 0.3

    for b in range(n_branches):
        # Upward curving branches
        angle = -math.pi/2 + (b - n_branches/2) * 0.3 + 0.1 * math.sin(ph + b)
        pts = [(cx, cy)]
        for step in range(int(10 + 15 * score)):
            t = step / max(1, (10 + 15 * score))
            angle += 0.15 * math.sin(ph * flow_speed + step * 0.7 + b)
            length = int(ART_SIZE * 0.04)
            nx = pts[-1][0] + int(length * math.cos(angle))
            ny = pts[-1][1] + int(length * math.sin(angle))
            pts.append((nx, ny))
        col = _lerp(primary, accent, b / max(1, n_branches))
        lw = max(2, int(6 * score * (1 - b / (2 * max(1, n_branches)))))
        if len(pts) > 1:
            draw.line(pts, fill=_c(col, int(200 * score)), width=lw)
        # Blossom circles at branch tips
        if len(pts) > 2:
            tx, ty = pts[-1]
            for petal in range(5):
                pa = 2 * math.pi * petal / 5 + ph
                pr = int(8 + 10 * score)
                px = tx + int(pr * math.cos(pa))
                py = ty + int(pr * math.sin(pa))
                draw.ellipse([px-pr//2, py-pr//2, px+pr//2, py+pr//2],
                             fill=_c(sparkle, int(160 * score)))

def _amplify_african(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[African — Amplify] Kente-inspired radiating strip weave.
    Bold geometric strips radiating outward with alternating colors
    and zigzag energy lines."""
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    ph = phase * 0.18
    n_strips = int(6 + 10 * score)

    for s in range(n_strips):
        angle = 2 * math.pi * s / n_strips + ph * 0.3
        # Wide strip from center outward
        strip_w = max(4, int(14 * score))
        length = int(ART_SIZE * 0.45 * score)
        # Strip as a series of alternating colored rectangles
        n_segments = int(4 + 6 * score)
        for seg in range(n_segments):
            t = seg / max(1, n_segments)
            r_start = int(ART_SIZE * 0.05 + length * t)
            r_end = r_start + length // max(1, n_segments)
            x1 = cx + int(r_start * math.cos(angle))
            y1 = cy + int(r_start * math.sin(angle))
            x2 = cx + int(r_end * math.cos(angle))
            y2 = cy + int(r_end * math.sin(angle))
            col = primary if seg % 2 == 0 else accent
            draw.line([(x1,y1),(x2,y2)], fill=_c(col, int(220 * score)),
                      width=strip_w)
        # Zigzag accent at tip
        tip_x = cx + int(length * math.cos(angle))
        tip_y = cy + int(length * math.sin(angle))
        zz_pts = [(tip_x, tip_y)]
        for z in range(4):
            zz_pts.append((tip_x + int(10*math.cos(angle+z*0.8)),
                           tip_y + int(10*math.sin(angle+z*0.8)) - z*6))
        if len(zz_pts) > 1:
            draw.line(zz_pts, fill=_c(sparkle, int(200*score)), width=2)

def _amplify_caucasian(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Caucasian — Amplify] Neurographic art — flowing curves that
    intersect and round at crossings. Expansive, organic growth pattern."""
    ph = phase * 0.15
    n_lines = int(6 + 10 * score)
    flow_speed = 1.0 + cv_feats.get("flow_magnitude", 0) * 0.2

    random.seed(int(phase * 3) % (2**31))
    for line in range(n_lines):
        pts = [(random.randint(50, ART_SIZE-50), random.randint(50, ART_SIZE-50))]
        angle = random.random() * 2 * math.pi
        for step in range(int(15 + 20 * score)):
            angle += 0.4 * math.sin(ph * flow_speed + step * 0.5 + line)
            length = int(ART_SIZE * 0.025)
            nx = pts[-1][0] + int(length * math.cos(angle))
            ny = pts[-1][1] + int(length * math.sin(angle))
            pts.append((max(0, min(ART_SIZE, nx)), max(0, min(ART_SIZE, ny))))
        col = _lerp(primary, accent, line / max(1, n_lines))
        lw = max(2, int(5 * score))
        if len(pts) > 1:
            draw.line(pts, fill=_c(col, int(200 * score)), width=lw)
        # Rounded intersection dots
        for pt in pts[::3]:
            r = max(3, int(6 * score))
            draw.ellipse([pt[0]-r, pt[1]-r, pt[0]+r, pt[1]+r],
                         fill=_c(sparkle, int(100 * score)))

def _amplify_hispanic(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Hispanic — Amplify] Papel picado burst — radiating cut-paper
    patterns with scalloped edges and dynamic symmetry."""
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    ph = phase * 0.18
    n_rays = int(8 + 16 * score)

    for i in range(n_rays):
        angle = 2 * math.pi * i / n_rays + ph * 0.4
        length = int(ART_SIZE * (0.15 + 0.30 * score))
        # Ray with scalloped edges (sine wave along the ray)
        pts_left, pts_right = [], []
        n_segs = int(10 + 15 * score)
        for seg in range(n_segs):
            t = seg / max(1, n_segs)
            r = int(ART_SIZE * 0.04 + length * t)
            # Scallop width oscillates
            scallop = int(8 * score * math.sin(t * math.pi * 4 + ph))
            perp = angle + math.pi / 2
            bx = cx + int(r * math.cos(angle))
            by = cy + int(r * math.sin(angle))
            pts_left.append((bx + int(scallop * math.cos(perp)),
                             by + int(scallop * math.sin(perp))))
            pts_right.append((bx - int(scallop * math.cos(perp)),
                              by - int(scallop * math.sin(perp))))
        pts = pts_left + pts_right[::-1]
        if len(pts) > 2:
            col = primary if i % 3 != 0 else accent
            draw.polygon(pts, fill=_c(col, int(140 * score)),
                         outline=_c(accent, int(200 * score)), width=1)

def _amplify_indian(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Indian — Amplify] Holi explosion — radial color splash with
    upward-arcing particle streams and concentric celebration rings."""
    cx, cy = ART_SIZE // 2, int(ART_SIZE * 0.55)
    ph = phase * 0.20
    n_streams = int(8 + 12 * score)

    # Upward particle streams
    for s in range(n_streams):
        angle = -math.pi/2 + (s - n_streams/2) * 0.25 + 0.2 * math.sin(ph + s)
        pts = [(cx + int(30 * (s - n_streams/2) * 0.3), cy)]
        for step in range(int(8 + 12 * score)):
            t = step / max(1, (8 + 12 * score))
            angle += 0.1 * math.sin(ph + step + s * 0.7)
            length = int(ART_SIZE * 0.035)
            nx = pts[-1][0] + int(length * math.cos(angle))
            ny = pts[-1][1] + int(length * math.sin(angle))
            pts.append((nx, ny))
            # Color splash dots along stream
            if step % 2 == 0:
                sz = max(3, int(10 * score * (1 - t)))
                col = _lerp(primary, sparkle, t) if s % 2 == 0 else _lerp(accent, sparkle, t)
                draw.ellipse([nx-sz, ny-sz, nx+sz, ny+sz], fill=_c(col, int(200 * score)))
        col = _lerp(primary, accent, s / max(1, n_streams))
        lw = max(1, int(3 * score))
        if len(pts) > 1:
            draw.line(pts, fill=_c(col, int(160 * score)), width=lw)

    # Celebration rings at top
    for ring in range(int(2 + 3 * score)):
        r = int(ART_SIZE * (0.05 + ring * 0.08))
        ry = int(ART_SIZE * 0.25)
        draw.arc([cx-r, ry-r, cx+r, ry+r], start=0, end=360,
                 fill=_c(accent, int(120 * score)), width=max(2, int(4*score)))

def _amplify_middle_eastern(draw, primary, accent, sparkle, phase, score, cv_feats):
    """[Middle Eastern — Amplify] Illuminated manuscript burst — radiating
    vine scrolls with leaf-shaped terminations and gold accents."""
    cx, cy = ART_SIZE // 2, ART_SIZE // 2
    ph = phase * 0.16
    n_vines = int(6 + 8 * score)

    for v in range(n_vines):
        base_angle = 2 * math.pi * v / n_vines + ph * 0.3
        pts = [(cx, cy)]
        angle = base_angle
        for step in range(int(12 + 18 * score)):
            t = step / max(1, (12 + 18 * score))
            # Gentle S-curve
            angle += 0.2 * math.sin(ph + step * 0.6 + v)
            length = int(ART_SIZE * 0.03)
            nx = pts[-1][0] + int(length * math.cos(angle))
            ny = pts[-1][1] + int(length * math.sin(angle))
            pts.append((nx, ny))
            # Leaf at intervals
            if step % 4 == 0 and step > 0:
                leaf_angle = angle + math.pi / 3
                lx = nx + int(12 * score * math.cos(leaf_angle))
                ly = ny + int(12 * score * math.sin(leaf_angle))
                leaf_pts = [(nx, ny),
                            (nx + int(8*math.cos(leaf_angle-0.3)), ny + int(8*math.sin(leaf_angle-0.3))),
                            (lx, ly),
                            (nx + int(8*math.cos(leaf_angle+0.3)), ny + int(8*math.sin(leaf_angle+0.3)))]
                col = _lerp(primary, accent, t)
                draw.polygon(leaf_pts, fill=_c(col, int(160 * score)))
        col = _lerp(primary, accent, v / max(1, n_vines))
        lw = max(2, int(4 * score))
        if len(pts) > 1:
            draw.line(pts, fill=_c(col, int(200 * score)), width=lw)

# =====================================================================
# LAYER 2: NEUTRAL PATTERNS — Balanced kinetic waves
# =====================================================================

def _balance_pattern(draw, primary, accent, sparkle, phase, score, cv_feats, race):
    """[Neutral — Balance] Gentle wave field adapted per race."""
    ph = phase * 0.10
    n_bands = int(8 + 10 * score)
    race_key = race.lower().strip() if race else ""
    # Different wave parameters per race
    if "asian" in race_key:
        freq, amp = 1.5, 25  # gentle
    elif "african" in race_key or "black" in race_key:
        freq, amp = 2.0, 30  # rhythmic
    elif "hispanic" in race_key or "latino" in race_key:
        freq, amp = 2.5, 28  # lively
    elif "indian" in race_key:
        freq, amp = 1.8, 22  # flowing
    elif "middle eastern" in race_key:
        freq, amp = 1.2, 20  # serene
    else:
        freq, amp = 1.6, 24  # balanced

    for b in range(n_bands):
        t = b / max(1, n_bands)
        pts = []
        for x in range(0, ART_SIZE + 1, 3):
            diag = (x + ART_SIZE * t)
            y = int(ART_SIZE * t + amp * score * math.sin(
                2 * math.pi * diag / ART_SIZE * freq + ph))
            pts.append((x, y))
        col = _lerp(primary, accent, t)
        lw = max(1, int(1 + 3 * t * score))
        if len(pts) > 1:
            draw.line(pts, fill=_c(col, int(100 + 120 * score)), width=lw)

# =====================================================================
# PATTERN DISPATCHER
# =====================================================================

_SOOTHE_FN = {
    "asian":          _soothe_asian,
    "african":        _soothe_african,
    "black":          _soothe_african,
    "caucasian":      _soothe_caucasian,
    "white":          _soothe_caucasian,
    "hispanic":       _soothe_hispanic,
    "latino hispanic":_soothe_hispanic,
    "indian":         _soothe_indian,
    "middle eastern": _soothe_middle_eastern,
}

_AMPLIFY_FN = {
    "asian":          _amplify_asian,
    "african":        _amplify_african,
    "black":          _amplify_african,
    "caucasian":      _amplify_caucasian,
    "white":          _amplify_caucasian,
    "hispanic":       _amplify_hispanic,
    "latino hispanic":_amplify_hispanic,
    "indian":         _amplify_indian,
    "middle eastern": _amplify_middle_eastern,
}

# =====================================================================
# LAYER 4 & 5: Sparkles & Vignette
# =====================================================================

def _draw_sparkles(draw, sparkle_col, phase, score):
    ph = phase * 0.25
    n = int(10 + score * 60)
    random.seed(int(phase * 7) % (2**31))
    for _ in range(n):
        x = random.randint(10, ART_SIZE - 10)
        y = random.randint(10, ART_SIZE - 10)
        r = random.randint(1, max(2, int(4 * score)))
        draw.ellipse([x-r, y-r, x+r, y+r],
                     fill=_c(sparkle_col, random.randint(80, 200)))

def _draw_vignette(img):
    vig = Image.new("RGBA", (ART_SIZE, ART_SIZE), (0, 0, 0, 0))
    d   = ImageDraw.Draw(vig)
    for s in range(30):
        t = s / 30
        r = int(ART_SIZE * 0.5 * (1 - t))
        d.ellipse([ART_SIZE//2-r, ART_SIZE//2-r, ART_SIZE//2+r, ART_SIZE//2+r],
                  outline=(0, 0, 0, int(150 * (t ** 2))), width=12)
    return Image.alpha_composite(img, vig)

# =====================================================================
# Gaussian blur layer for stressed — soft dreamy quality
# =====================================================================

def _apply_therapeutic_blur(img, intention, score):
    """[Stressed → soft Gaussian blur for dreamy containment.
     Happy → no blur, keep sharp and energetic.]"""
    if intention == "soothe":
        blur_radius = max(0.5, 2.0 * (1.0 - score))  # less confident = more blur
        return img.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    return img

# =====================================================================
# INTENTION MESSAGES
# =====================================================================

INTENTION_MESSAGES = {
    "soothe":  "🌊 Breathing calm into you with gentle symmetry",
    "amplify": "✨ Celebrating your energy — let it radiate!",
    "balance": "⚡ Gentle waves of balanced rhythm",
}

# =====================================================================
# MASTER RENDERER
# =====================================================================

def generate_art(emotions_list, race, cv_feats=None):
    """
    Generate therapeutic art driven by emotion, race, and CV features.

    emotions_list: [{label, score}, ...]
    race: string
    cv_feats: dict from extract_all_cv_features() or None
    """
    global art_phase
    art_phase += 1

    if cv_feats is None:
        cv_feats = {}

    if not emotions_list:
        emotions_list = [{"label": "Neutral", "score": 0.5}]

    top_emo   = emotions_list[0]["label"]
    top_score = emotions_list[0]["score"]

    # ── Determine therapeutic intention ──
    target = THERAPEUTIC_TARGETS.get(top_emo, THERAPEUTIC_TARGETS["Neutral"])
    intention, bg_dark, bg_light, base_primary, base_accent, base_sparkle = target

    # ── Apply race-specific shade shifting ──
    primary, accent, _ = get_race_shaded_palette(base_primary, base_accent, bg_light, race)
    sparkle = _hue_shift(base_sparkle, *RACE_SHADE_MAP.get(
        race.lower().strip() if race else "", (0, 1.0, 1.0)))

    # ── Mixed emotion blending ──
    if len(emotions_list) > 1:
        sec = emotions_list[1]
        if sec["score"] > 0.15:
            weight2 = sec["score"] / (top_score + sec["score"])
            sec_target = THERAPEUTIC_TARGETS.get(sec["label"], THERAPEUTIC_TARGETS["Neutral"])
            sec_primary = sec_target[3]
            sec_accent  = sec_target[4]
            sec_p, sec_a, _ = get_race_shaded_palette(sec_primary, sec_accent, bg_light, race)
            primary = _lerp(primary, sec_p, weight2 * 0.4)
            accent  = _lerp(accent,  sec_a, weight2 * 0.4)

    # ── Optical flow modulates animation speed ──
    flow_mag = cv_feats.get("flow_magnitude", 0.0)
    effective_phase = art_phase + int(flow_mag * 2)

    # ── Build canvas ──
    img = Image.new("RGBA", (ART_SIZE, ART_SIZE), (0, 0, 0, 255))
    _draw_bg(img, bg_dark, bg_light, effective_phase, intention)
    draw = ImageDraw.Draw(img)

    # ── Midground: race-specific pattern ──
    race_key = race.lower().strip() if race else ""

    if intention == "soothe":
        fn = _SOOTHE_FN.get(race_key, _soothe_caucasian)
        fn(draw, primary, accent, sparkle, effective_phase, top_score, cv_feats)
    elif intention == "amplify":
        fn = _AMPLIFY_FN.get(race_key, _amplify_caucasian)
        fn(draw, primary, accent, sparkle, effective_phase, top_score, cv_feats)
    else:
        _balance_pattern(draw, primary, accent, sparkle, effective_phase, top_score, cv_feats, race)

    # ── Sparkle + vignette + therapeutic blur ──
    _draw_sparkles(draw, sparkle, effective_phase, top_score)
    img = _draw_vignette(img)
    img = _apply_therapeutic_blur(img, intention, top_score)

    img_rgb = img.convert("RGB")
    buf = io.BytesIO()
    img_rgb.save(buf, format="JPEG", quality=90)
    return buf.getvalue(), INTENTION_MESSAGES.get(intention, "")


print("✅ Therapeutic Art Engine v13 ready")
print("   • Race-differentiated shades (HSV shifting per race)")
print("   • 6 unique soothe patterns (Enso, Adinkra, Fibonacci, Ojo de Dios, Rangoli, Arabesque)")
print("   • 6 unique amplify patterns (Cherry Blossom, Kente, Neurographic, Papel Picado, Holi, Vine Scroll)")
print("   • CV features drive: complexity, animation speed, detail, blur, curvature")



✅ Therapeutic Art Engine v13 ready
   • Race-differentiated shades (HSV shifting per race)
   • 6 unique soothe patterns (Enso, Adinkra, Fibonacci, Ojo de Dios, Rangoli, Arabesque)
   • 6 unique amplify patterns (Cherry Blossom, Kente, Neurographic, Papel Picado, Holi, Vine Scroll)
   • CV features drive: complexity, animation speed, detail, blur, curvature


In [12]:
# ============================================================
# Cell 8C: Art Preview — test all 7 emotions × 6 races
# ============================================================
import ipywidgets as widgets
from IPython.display import display

PREVIEW_RACES = ["asian", "african", "caucasian", "hispanic", "indian", "middle eastern"]
EMOTIONS_ALL  = ["Happy", "Sad", "Angry", "Fear", "Surprise", "Disgust", "Neutral"]

print("🎨 Generating art preview grid (7 emotions × 6 races)...")

rows = []
# Header row
header = [widgets.Label(value="", layout=widgets.Layout(width="60px"))]
for race in PREVIEW_RACES:
    header.append(widgets.Label(value=race.title(),
        layout=widgets.Layout(width="90px", align_self="center")))
rows.append(widgets.HBox(header))

for emo in EMOTIONS_ALL:
    row_widgets = [widgets.Label(value=emo, layout=widgets.Layout(width="60px"))]
    for race in PREVIEW_RACES:
        mock = [{"label": emo, "score": 0.80}]
        art_bytes, _ = generate_art(mock, race, cv_feats={
            "edge_density": 0.05, "sift_count": 80,
            "flow_magnitude": 0.5, "focus_measure": 150,
            "curvature": 5.0, "mean_luminance": 0.5,
            "estimated_distance_cm": 60, "skin_ratio": 0.6,
        })
        w = widgets.Image(value=art_bytes, format='jpeg', width=85, height=85)
        row_widgets.append(w)
    rows.append(widgets.HBox(row_widgets,
        layout=widgets.Layout(margin="2px 0")))

display(widgets.VBox(rows, layout=widgets.Layout(align_items="center")))
print("✅ Preview: each column = different race shade & pattern")



🎨 Generating art preview grid (7 emotions × 6 races)...


✅ Preview: each column = different race shade & pattern


In [15]:
# ============================================================
# Cell 9: Immersive Live Inference + Therapeutic Art Loop (v13)
#
# UPGRADES from v12:
#  - Runs ALL 17 CV techniques each frame on detected faces
#  - Passes cv_feats dict to art engine for CV-driven art
#  - Optical Flow tracked between frames on landmarks
#  - Appearance Matching assigns persistent face IDs
#  - Edge/SIFT/Boundary metrics displayed in status bar
#
# CV Techniques active in this loop:
#  [1]  Image Formation      — face alignment via affine warp
#  [2]  Image Sensing        — noise estimation
#  [3]  Binary Images        — Otsu face mask
#  [4]  Image Processing I   — CLAHE, gamma, white balance
#  [5]  Image Processing II  — unsharp mask
#  [6]  Edge Detection       — Sobel + Canny
#  [7]  Boundary Detection   — contour analysis
#  [8]  SIFT Detector        — keypoint density
#  [9]  Face Detection       — MTCNN + MediaPipe landmarks
#  [10] Radiometry           — mean reflectance
#  [11] Shape from Shading   — surface normals
#  [12] Depth from Defocus   — Laplacian variance
#  [13] Optical Flow         — Lucas-Kanade on landmarks
#  [14] Camera Calibration   — focal length estimation
#  [15] Image Segmentation   — skin color detection
#  [16] Appearance Matching  — histogram face ReID
#  [17] Object Tracking      — centroid tracker
# ============================================================

ART_REGEN_FRAMES = 8

# ── Camera init ────────────────────────────────────────────────
print("📷 Starting camera...")
_result = _init_camera()
if _result != "OK":
    raise RuntimeError(f"Camera failed: {_result}. Re-run this cell.")
print("✅ Camera ready.")

emotion_history_per_face.clear()
race_history_per_face.clear()
art_phase = 0

# Reset CV tracking state
_prev_gray = None
_prev_landmarks = None
_face_histograms.clear()
_face_centroids.clear()
_next_face_id = 0

# ── UI Layout ──────────────────────────────────────────────────
cam_widget = widgets.Image(format="jpeg", width=640, height=512)
art_widget = widgets.Image(format="jpeg", width=512, height=512)

cam_lbl = widgets.HTML(value="<b style='font-size:14px'>📷 Live Feed</b>")
art_lbl = widgets.HTML(value="<b style='font-size:14px'>🎨 Therapeutic Art</b>")
intention_lbl = widgets.HTML(value="<i style='color:#888'>Initialising...</i>")
cv_info_lbl = widgets.HTML(value="<i style='color:#666'>CV metrics loading...</i>")
status_lbl = widgets.Label(value="🎥 Starting...")

cam_panel = widgets.VBox([cam_lbl, cam_widget],
                         layout=widgets.Layout(align_items="center", height="580px"))
art_panel = widgets.VBox([art_lbl, art_widget, intention_lbl, cv_info_lbl],
                         layout=widgets.Layout(align_items="center", height="580px"))

display(status_lbl)
display(widgets.HBox([cam_panel, art_panel],
        layout=widgets.Layout(justify_content="center", gap="28px")))
inject_stop_button()

# ── Seed first art frame ──────────────────────────────────────
_init_bytes, _init_msg = generate_art([{"label": "Neutral", "score": 0.5}], "")
art_widget.value   = _init_bytes
intention_lbl.value = f"<i style='color:#888'>{_init_msg}</i>"

print("🎥 Starting immersive live feed with 17 CV techniques...")

# ── Loop state ─────────────────────────────────────────────────
frames_since_regen = 0
top_emotion   = "Neutral"
top_score     = 0.5
dominant_race = ""
current_emotions = [{"label": "Neutral", "score": 0.5}]
current_cv_feats = {}

# ── Main loop ──────────────────────────────────────────────────
while True:
    try:
        frame = capture_frame_from_browser()
        if frame is None:
            status_lbl.value = "⏹ Stopped."
            break

        frame_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  # for Optical Flow

        # ── [CV Technique 9] Face Detection (MTCNN) ──────────
        boxes, _ = mtcnn.detect(frame_rgb)

        if boxes is not None:
            all_results = None
            race_scores = {}

            for box in boxes:
                x1, y1, x2, y2 = [max(0, int(b)) for b in box]
                raw_crop = frame_rgb[y1:y2, x1:x2]
                if raw_crop is None or raw_crop.size == 0:
                    continue

                # ── Enhancement pipeline [CV 1,4,5,9] ────────
                try:
                    processed_crop, quality = preprocess_face(raw_crop)
                    inference_crop = processed_crop if processed_crop is not None \
                                     else preprocess_face_fallback(raw_crop)
                except Exception:
                    inference_crop = np.ascontiguousarray(raw_crop, dtype=np.uint8)
                    quality = 0.0

                # ── Extract ALL CV features [CV 3,6,7,8,10-16] ─
                try:
                    current_cv_feats = extract_all_cv_features(
                        inference_crop, (x1, y1, x2, y2), frame_gray)
                except Exception:
                    current_cv_feats = {}

                # ── [CV Technique 13] Optical Flow ────────────
                try:
                    landmarks_px, _ = get_landmark_quality(inference_crop)
                    if landmarks_px is not None:
                        # Scale landmarks to frame coordinates
                        h_crop, w_crop = inference_crop.shape[:2]
                        scaled_lmks = [
                            (int(lm[0] / max(1,w_crop) * (x2-x1) + x1),
                             int(lm[1] / max(1,h_crop) * (y2-y1) + y1))
                            for lm in landmarks_px
                        ]
                        flow_feats = compute_optical_flow(frame_gray, scaled_lmks)
                        current_cv_feats.update(flow_feats)
                except Exception:
                    current_cv_feats["flow_magnitude"] = 0.0

                # ── [CV Technique 17] Object Tracking ─────────
                try:
                    fid = current_cv_feats.get("face_id", 0)
                    track_face_centroid((x1, y1, x2, y2), fid)
                except Exception:
                    pass

                pil_crop = Image.fromarray(inference_crop)

                # ── Emotion inference [CV 9] ──────────────────
                try:
                    all_results = emotion_model(pil_crop, top_k=7)
                    all_results = sorted(all_results, key=lambda r: r["score"], reverse=True)
                    top = all_results[0]
                    top_score   = top["score"]
                    fid = current_cv_feats.get("face_id", 0)
                    top_emotion = smooth_emotion(
                        EMOTION_LABELS.get(top["label"], top["label"]), face_id=fid)
                    current_emotions = [{"label": top_emotion, "score": top_score}]
                    if len(all_results) > 1:
                        sec_label = EMOTION_LABELS.get(all_results[1]["label"], all_results[1]["label"])
                        current_emotions.append({"label": sec_label, "score": all_results[1]["score"]})
                except Exception:
                    pass

                # ── Race inference ────────────────────────────
                try:
                    race_results = race_model(pil_crop, top_k=6)
                    race_scores  = {r["label"].lower(): r["score"] * 100 for r in race_results}
                    top_race = max(race_results, key=lambda r: r["score"])
                    if top_race["score"] >= RACE_CONF_THRESHOLD:
                        dominant_race = smooth_race(top_race["label"].title(), face_id=fid if "fid" in dir() else current_cv_feats.get("face_id", 0))
                except Exception:
                    pass

                # ── Draw bounding box ─────────────────────────
                box_color = (0, 255, 0) if quality >= QUALITY_THRESHOLD else (140, 140, 140)
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                if top_score >= EMOTION_CONF_THRESHOLD:
                    race_tag = f" | {dominant_race}" if dominant_race else ""
                    fid_tag = f" [ID:{current_cv_feats.get('face_id', '?')}]"
                    cv2.putText(frame,
                                f"{top_emotion} ({top_score:.0%}){race_tag}{fid_tag}",
                                (x1, max(y1 - 10, 70)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.65, box_color, 2)

            if all_results:
                frame = draw_emotion_bar(frame, sorted(all_results, key=lambda r: r["label"]))
            if race_scores:
                frame = draw_race_bar(frame, race_scores)

        # Update webcam
        cam_widget.value = frame_to_jpeg(frame)

        # ── Art regen every N frames ──────────────────────────
        if frames_since_regen >= ART_REGEN_FRAMES:
            try:
                art_bytes, msg = generate_art(
                    current_emotions,
                    dominant_race.lower() if dominant_race else "",
                    cv_feats=current_cv_feats
                )
                art_widget.value    = art_bytes
                intention_lbl.value = f"<i style='color:#aaa;font-size:13px'>{msg}</i>"
                frames_since_regen  = 0

                # Update CV info display
                edge_d  = current_cv_feats.get("edge_density", 0)
                sift_n  = current_cv_feats.get("sift_count", 0)
                flow_m  = current_cv_feats.get("flow_magnitude", 0)
                focus   = current_cv_feats.get("focus_measure", 0)
                skin_r  = current_cv_feats.get("skin_ratio", 0)
                curv    = current_cv_feats.get("curvature", 0)
                cv_info_lbl.value = (
                    f"<span style='color:#777;font-size:11px'>"
                    f"Edge:{edge_d:.2f} | SIFT:{sift_n} | Flow:{flow_m:.1f} | "
                    f"Focus:{focus:.0f} | Skin:{skin_r:.0%} | Curv:{curv:.1f}"
                    f"</span>"
                )
            except Exception as ae:
                status_lbl.value = f"Art error: {ae}"
        else:
            frames_since_regen += 1

        # Status bar
        target = THERAPEUTIC_TARGETS.get(top_emotion, THERAPEUTIC_TARGETS["Neutral"])
        status_lbl.value = (
            f"🎥 {top_emotion} ({top_score:.0%}) | {dominant_race or '—'} | "
            f"🎨 {target[0]} | phase:{art_phase}"
        )

    except Exception as e:
        status_lbl.value = f"❌ Error: {e}"
        break

print("✅ Done.")





📷 Starting camera...
✅ Camera ready.


Label(value='🎥 Starting...')

🎥 Starting immersive live feed with 17 CV techniques...
✅ Done.
